In [1]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [2]:
from pathlib import Path


from IPython.display import display

import polars as pl
import duckdb as dd


from tabseq_torch.dl import (
    DataCatalog,
    read_json,
)
from tabseq_torch.dl.integration_tests import ds_memory_usage_tx, n_rows_expected

In [3]:
print(pl.__version__)

1.39.3


In [ ]:
tx_path = Path(
    "/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/train_transactions_contest"
)
print(tx_path.exists())
for p in tx_path.parent.iterdir():
    print(p)
    lf_tx = pl.scan_parquet(tx_path)
display(lf_tx.collect_schema())
dc = DataCatalog(
    src_tabts_path=tx_path,
    cat_path=tx_path.parent / "cat",
    meta_path=tx_path.parent / "meta",
    targets_path=tx_path.parent / "flag.parquet",
)
dc.print_setnames()
setname = "allcat"
groupby = "app_id"
display(pl.scan_parquet(dc.cat(setname=setname)).collect_schema())
cat_columns = list(read_json(dc.catnum_json(setname=setname)).keys())
print(cat_columns)
print(len(cat_columns))
num_columns = ["amnt", "days_before", "transaction_number", "hour_diff"]
print(num_columns)

True
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/encoded
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/__MACOSX
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/flag.parquet
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/cat
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/tb_logs
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/meta
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/train_transactions_contest


Schema([('app_id', Int32),
        ('amnt', Float64),
        ('currency', Int32),
        ('operation_kind', Int32),
        ('card_type', Int32),
        ('operation_type', Int32),
        ('operation_type_group', Int32),
        ('ecommerce_flag', Int32),
        ('payment_system', Int32),
        ('income_flag', Int32),
        ('mcc', Int32),
        ('country', Int32),
        ('city', Int32),
        ('mcc_category', Int32),
        ('day_of_week', Int32),
        ('hour', Int32),
        ('days_before', Int32),
        ('weekofyear', Int32),
        ('hour_diff', Int64),
        ('transaction_number', Int32),
        ('__index_level_0__', Int64)])

allcat
test


Schema([('app_id', Int32),
        ('currency', Int32),
        ('operation_kind', Int32),
        ('card_type', Int32),
        ('operation_type', Int32),
        ('operation_type_group', Int32),
        ('ecommerce_flag', Int32),
        ('payment_system', Int32),
        ('income_flag', Int32),
        ('mcc', Int32),
        ('country', Int32),
        ('city', Int32),
        ('mcc_category', Int32),
        ('day_of_week', Int32),
        ('hour', Int32),
        ('weekofyear', Int32)])

['currency', 'operation_kind', 'card_type', 'operation_type', 'operation_type_group', 'ecommerce_flag', 'payment_system', 'income_flag', 'mcc', 'country', 'city', 'mcc_category', 'day_of_week', 'hour', 'weekofyear']
15
['amnt', 'hour_diff']


In [6]:
%%time
n_rows_loaded = ds_memory_usage_tx(
    dc,
    num_chunk_per_worker=10,
    memlimit_gb=0.2,
    cat_columns_to_select=cat_columns,
    num_columns_to_select=num_columns,
    dryrun=False,
    read_groups=True,
    setname=setname,
)
n_rows_except_ = n_rows_expected(dc.src)
n_rows_except_ == n_rows_loaded

_num_chunk_per_worker=10
i=0
loading: memory usage: 1862.9MB
chunk memorys: 14.7MB
i=1
loading: memory usage: 1862.9MB
chunk memorys: 14.7MB
i=2
loading: memory usage: 1862.9MB
chunk memorys: 14.7MB
i=3
loading: memory usage: 1862.9MB
chunk memorys: 14.6MB
i=4
loading: memory usage: 1864.8MB
chunk memorys: 14.7MB
i=5
loading: memory usage: 1865.1MB
chunk memorys: 14.7MB
i=6
loading: memory usage: 1865.1MB
chunk memorys: 14.7MB
i=7
loading: memory usage: 1865.1MB
chunk memorys: 14.7MB
i=8
loading: memory usage: 1865.1MB
chunk memorys: 14.7MB
i=9
loading: memory usage: 1865.1MB
chunk memorys: 14.8MB
num rows: 270450065
CPU times: user 13min 56s, sys: 1min 10s, total: 15min 7s
Wall time: 12min 2s


True